# Collect Cosmos-Policy inputs across paired prompts

Build a pool of policy-model inputs for downstream LQR / SVD analysis.

Setup
- Suite: `libero_10`, task `0` ("put both the alphabet soup and the tomato sauce in the basket").
- Use the first `N_EPISODES = 10` saved initial states.
- For each initial state, run one rollout under each of two prompts:
  - **original**:  `"put both the alphabet soup and the tomato sauce in the basket"`
  - **disturbed**: original + `" the cream cheese, ketchup, orange juice, milk, and butter are also on the table."`
  (matches the phrasing that `PromptDistractUnusedObjects` with `mode='neutral'` would produce)
- At every inference call (i.e. each time the open-loop action queue runs dry), record the
  inputs that `get_action` consumes: `primary_image`, `wrist_image`, `proprio`.
- Save every inference across every rollout into a single `.npz` (one pool, not split by prompt).
  Per-row tags (`episode_idx`, `prompt_idx`, `inference_idx`) let downstream code re-partition.

Mirrors the rollout structure of `notebooks/stress_test/01_robot_color.ipynb` but strips the
stress-test infrastructure — only the two prompts above vary.

In [ ]:
import sys; sys.path.insert(0, '../..')
from _setup import setup_env
setup_env()

import os
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')

In [ ]:
import json
import time
from collections import deque
from pathlib import Path

import numpy as np

from libero.libero import benchmark
from cosmos_policy.experiments.robot.libero.libero_utils import (
    get_libero_env, get_libero_dummy_action,
)
from cosmos_policy.experiments.robot.libero.run_libero_eval import (
    PolicyEvalConfig, prepare_observation, TASK_MAX_STEPS,
)
from cosmos_policy.experiments.robot.cosmos_utils import (
    get_action, get_model, load_dataset_stats, init_t5_text_embeddings_cache,
)

## 1. Config

In [ ]:
SUITE_NAME  = 'libero_10'
TASK_ID     = 0
N_EPISODES  = 10
RESOLUTION  = 256

PROMPT_ORIGINAL  = 'put both the alphabet soup and the tomato sauce in the basket'
PROMPT_DISTURBED = (
    'put both the alphabet soup and the tomato sauce in the basket.'
    ' the cream cheese, ketchup, orange juice, milk, and butter are also on the table.'
)
PROMPTS = [('original', PROMPT_ORIGINAL), ('disturbed', PROMPT_DISTURBED)]

OUT_DIR = Path('notebooks/lqr/inputs/policy_inputs') / f'{SUITE_NAME}__task{TASK_ID:02d}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_NPZ      = OUT_DIR / 'inputs.npz'
OUT_MANIFEST = OUT_DIR / 'manifest.json'

print(f'suite     : {SUITE_NAME}')
print(f'task      : {TASK_ID}')
print(f'episodes  : {N_EPISODES}')
print(f'prompts   :')
for n, p in PROMPTS:
    print(f'  [{n}] {p!r}')
print(f'output -> {OUT_NPZ.resolve()}')

## 2. Build env + load saved initial states

Same env construction as `06_libero_rollout.ipynb` — the env itself is identical across the
two prompts; only the text passed to the policy differs. We reuse one env and reset it per
rollout via `set_init_state`.

In [ ]:
task_suite = benchmark.get_benchmark_dict()[SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
init_states = task_suite.get_task_init_states(TASK_ID)
print(f'task.language: {task.language!r}')
print(f'init states available: {init_states.shape[0]}  (using the first {N_EPISODES})')
assert N_EPISODES <= init_states.shape[0], f'only {init_states.shape[0]} init states available'

env, task_desc_from_env = get_libero_env(task, 'cosmos', resolution=RESOLUTION)
max_env_steps = TASK_MAX_STEPS[SUITE_NAME]
print(f'env={type(env).__name__}  max_steps={max_env_steps}')
print(f'env-provided task_desc: {task_desc_from_env!r}')

## 3. Load the Cosmos-Policy checkpoint

Heavy one-time step. Config matches the README quick-start and the other LIBERO rollout
notebooks. First-time use of the **disturbed** prompt pays a one-time T5-11B load + embed
because it isn't in the prebaked cache.

In [ ]:
cfg = PolicyEvalConfig(
    config='cosmos_predict2_2b_480p_libero__inference_only',
    ckpt_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B',
    config_file='cosmos_policy/config/config.py',
    dataset_stats_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_dataset_statistics.json',
    t5_text_embeddings_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_t5_embeddings.pkl',
    use_wrist_image=True, use_proprio=True, normalize_proprio=True, unnormalize_actions=True,
    chunk_size=16, num_open_loop_steps=16, trained_with_image_aug=True,
    use_jpeg_compression=True, flip_images=True,
    num_denoising_steps_action=5,
    num_denoising_steps_future_state=1, num_denoising_steps_value=1,
    task_suite_name=SUITE_NAME,
)
dataset_stats = load_dataset_stats(cfg.dataset_stats_path)
init_t5_text_embeddings_cache(cfg.t5_text_embeddings_path)
model, _ = get_model(cfg)
print('model ready')

## 4. Rollout that records every policy input

Same Mode-A rollout loop as the other LIBERO notebooks. The only addition: every time the
action queue is empty we are about to call the policy — capture the `observation` dict
(`primary_image`, `wrist_image`, `proprio`) immediately before that call.

Note: `prepare_observation(..., resize_size=224)` does not actually resize on this code
path; it just packs the raw `agentview_image` and `robot0_eye_in_hand_image` (both
`RESOLUTION × RESOLUTION × 3` uint8) plus the 9-D proprio (gripper_qpos:2 + eef_pos:3 +
eef_quat:4). Resize happens inside `get_action` via `prepare_images_for_model`, so the
inputs we capture here are exactly what we'd feed to `get_action` on replay.

In [ ]:
def policy_fn(obs, desc):
    out = get_action(
        cfg, model, dataset_stats, obs, desc,
        num_denoising_steps_action=cfg.num_denoising_steps_action,
        generate_future_state_and_value_in_parallel=True,
    )
    return out['actions']


def rollout_collect_inputs(env, init_state, task_desc, *, num_steps_wait=10):
    """Run one Mode-A rollout. Returns:
      - success (bool), env_steps (int)
      - inputs: list of dicts, one per inference call, each with
          {'primary_image', 'wrist_image', 'proprio'} as numpy arrays.
    """
    env.reset()
    obs = env.set_init_state(init_state)

    # Let the scene settle (matches run_libero_eval).
    for _ in range(num_steps_wait):
        obs, _, _, _ = env.step(get_libero_dummy_action(cfg.model_family))

    queue = deque(maxlen=cfg.num_open_loop_steps)
    inputs = []
    success = False
    t = 0
    while t < max_env_steps:
        if not queue:
            observation = prepare_observation(obs, resize_size=224, flip_images=cfg.flip_images)
            inputs.append({
                'primary_image': np.ascontiguousarray(observation['primary_image']),
                'wrist_image':   np.ascontiguousarray(observation['wrist_image']),
                'proprio':       np.asarray(observation['proprio'], dtype=np.float32),
            })
            actions = policy_fn(observation, task_desc)
            for a in actions[:cfg.num_open_loop_steps]:
                queue.append(np.asarray(a, dtype=np.float32))
        a = queue.popleft()
        obs, _, done, _ = env.step(a.tolist())
        if done:
            success = True
            break
        t += 1
    return success, t + num_steps_wait, inputs

## 5. Run all rollouts (10 episodes × 2 prompts) and accumulate inputs

In [ ]:
all_primary, all_wrist, all_proprio = [], [], []
all_episode_idx, all_prompt_idx, all_inference_idx = [], [], []
rollout_summaries = []

for ep in range(N_EPISODES):
    for prompt_idx, (prompt_name, prompt_text) in enumerate(PROMPTS):
        t0 = time.time()
        success, env_steps, inputs = rollout_collect_inputs(env, init_states[ep], prompt_text)
        dt = time.time() - t0

        for inf_idx, rec in enumerate(inputs):
            all_primary.append(rec['primary_image'])
            all_wrist.append(rec['wrist_image'])
            all_proprio.append(rec['proprio'])
            all_episode_idx.append(ep)
            all_prompt_idx.append(prompt_idx)
            all_inference_idx.append(inf_idx)

        tag = 'SUCCESS' if success else 'FAILURE'
        n_inf = len(inputs)
        print(f'ep {ep:2d}  prompt={prompt_name:9s}  {tag:7s}  steps={env_steps:4d}  inferences={n_inf:3d}  {dt:6.1f}s')
        rollout_summaries.append({
            'episode': ep,
            'prompt_idx': prompt_idx,
            'prompt_name': prompt_name,
            'success': bool(success),
            'env_steps': int(env_steps),
            'n_inferences': n_inf,
            'wall_time_s': dt,
        })

env.close()

primary_arr   = np.stack(all_primary,   axis=0)
wrist_arr     = np.stack(all_wrist,     axis=0)
proprio_arr   = np.stack(all_proprio,   axis=0)
episode_arr   = np.asarray(all_episode_idx,   dtype=np.int32)
prompt_arr    = np.asarray(all_prompt_idx,    dtype=np.int32)
inference_arr = np.asarray(all_inference_idx, dtype=np.int32)

print()
print(f'total inferences collected: {primary_arr.shape[0]}')
print(f'  primary_images: {primary_arr.shape}  dtype={primary_arr.dtype}')
print(f'  wrist_images:   {wrist_arr.shape}  dtype={wrist_arr.dtype}')
print(f'  proprios:       {proprio_arr.shape}  dtype={proprio_arr.dtype}')

## 6. Save

One `.npz` holds the full pool (images + proprio + per-row tags). A sibling `manifest.json`
stores the prompt texts, config, and per-rollout summaries so the npz alone is enough to
interpret each row.

In [ ]:
np.savez_compressed(
    OUT_NPZ,
    primary_images=primary_arr,
    wrist_images=wrist_arr,
    proprios=proprio_arr,
    episode_idx=episode_arr,
    prompt_idx=prompt_arr,
    inference_idx=inference_arr,
)

manifest = {
    'suite': SUITE_NAME,
    'task_id': TASK_ID,
    'n_episodes': N_EPISODES,
    'resolution': RESOLUTION,
    'prompts': [{'idx': i, 'name': n, 'text': p} for i, (n, p) in enumerate(PROMPTS)],
    'image_layout': 'HWC uint8, flip_images=True applied at capture time (same as get_action input)',
    'proprio_layout': 'concat(robot0_gripper_qpos[2], robot0_eef_pos[3], robot0_eef_quat[4]) -> shape (9,) float32',
    'total_inferences': int(primary_arr.shape[0]),
    'rollouts': rollout_summaries,
}
OUT_MANIFEST.write_text(json.dumps(manifest, indent=2))

print(f'wrote {OUT_NPZ}  ({OUT_NPZ.stat().st_size / 1e6:.1f} MB)')
print(f'wrote {OUT_MANIFEST}')

## 7. Summary

In [ ]:
from collections import Counter

succ_by_prompt = Counter()
inf_by_prompt = Counter()
for r in rollout_summaries:
    succ_by_prompt[r['prompt_name']] += int(r['success'])
    inf_by_prompt[r['prompt_name']]  += r['n_inferences']

for name, _ in PROMPTS:
    print(f'  prompt={name:9s}  success={succ_by_prompt[name]}/{N_EPISODES}  inferences={inf_by_prompt[name]}')
print(f'  total inferences pooled: {primary_arr.shape[0]}')
print(f'  file: {OUT_NPZ.resolve()}')